In [1]:
from notion_functions import NotionPage
from handler import Summarizer
from epub_functions import Book
import os
from dotenv import load_dotenv
from supporting_functions import load_json, save_json, create_book_content_html_and_serve_with_flask, get_parsed_html_content
from html_functions import KindleHTMLParser, create_html

In [2]:
book_name = "Invisible Empire"
folder = "./books/invisible-empire/"
book_source = folder + "book.epub"
highlights_source = folder + "highlights.html"
summary_instructions = "Focus on the anecdotes and the facts shared in the book."

In [3]:
chapters_to_skip = ["Cover"]
book = Book(book_source)
book_content = book.book_content
toc_details = book.toc_details
save_json(toc_details, folder + "toc.json")
file_details = book.files_in_epub
save_json(file_details, folder + "files.json")
highlights = KindleHTMLParser(highlights_source).highlights
save_json(highlights, folder + "highlights.json")
create_book_content_html_and_serve_with_flask(book_content, file_details, folder)



d:\Projects\Fun\summarizer\.venv\Lib\site-packages\ebooklib\epub.py:1395: UserWarning: In the future version we will turn default option ignore_ncx to True.
  warnings.warn('In the future version we will turn default option ignore_ncx to True.')
d:\Projects\Fun\summarizer\.venv\Lib\site-packages\ebooklib\epub.py:1423: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/xmlns:rootfile[@media-type]'
  for root_file in tree.findall('//xmlns:rootfile[@media-type]', namespaces={'xmlns': NAMESPACES['CONTAINERNS']}):


 * Serving Flask app 'supporting_functions'
 * Debug mode: off


 * Running on http://127.0.0.1:5500
Press CTRL+C to quit


In [8]:
toc = load_json(folder + "toc.json")
all_highlights = load_json(folder + "highlights.json")
notion_parent_page_id = os.getenv('NOTION_PARENT_PAGE_ID')
summarizer = Summarizer(summary_instructions)
notion_page = NotionPage(notion_parent_page_id, book_name)

def find_highlights_for_chapter(chapter_title, all_chapter_highlights):
    for chapter in all_chapter_highlights:
        if chapter['title'] == chapter_title:
            return chapter['highlights']
    return []

for item in toc:
    if item["type"] == "link":
        chapter = {"id": item["uid"], "title": item["title"], "content": get_parsed_html_content(book_content, item["href"])}
        highlights = find_highlights_for_chapter(item["title"], all_highlights)
        summary = summarizer.get_summary(chapter, highlights)
        notion_page.add_chapter_to_page(notion_page.page_id, summary, highlights)
    if item["type"] == "section":
        notion_page.add_section_heading_to_page(notion_page.page_id, item["title"])
        for subitem in item["links"]:
            chapter = {"id": subitem["uid"], "title": subitem["title"], "content": get_parsed_html_content(book_content, subitem["href"])}
            highlights = find_highlights_for_chapter(item["title"], all_highlights)
            summary = summarizer.get_summary(chapter, highlights)
            notion_page.add_chapter_to_page(notion_page.page_id, summary, highlights)

Creating Notion Page: Invisible Empire
Summarizing chapter: Cover
Adding Chapter to Notion Page: Cover
Summarizing chapter: Contents
Adding Chapter to Notion Page: Contents
Summarizing chapter: 1 BOUNTY
Adding Chapter to Notion Page: 1 BOUNTY
Summarizing chapter: 2 A WHOLE NEW WORLD
Adding Chapter to Notion Page: 2 A WHOLE NEW WORLD
Summarizing chapter: 3 SUPERSIZE ME
Adding Chapter to Notion Page: 3 SUPERSIZE ME
Summarizing chapter: 4 THE VIRUS IS US
Adding Chapter to Notion Page: 4 THE VIRUS IS US
Summarizing chapter: 5 A DEEP CONTROL
Adding Chapter to Notion Page: 5 A DEEP CONTROL
Summarizing chapter: 6 INVADERS, HITCH-HIKERS, SENTINELS, KILLERS
Adding Chapter to Notion Page: 6 INVADERS, HITCH-HIKERS, SENTINELS, KILLERS
Summarizing chapter: 7 A SPOTTY HISTORY OF THE SPECKLED MONSTER
Adding Chapter to Notion Page: 7 A SPOTTY HISTORY OF THE SPECKLED MONSTER
Summarizing chapter: 8 GUT FEELING
Adding Chapter to Notion Page: 8 GUT FEELING
Summarizing chapter: 9 A VIRUS VANISHES
Adding Ch